# Saved PIC analysis and manuscript comparisons
Run all cells after setting the configuration below. With no simulation data, the
notebook reports what is missing and skips plots; it does not fabricate results.
Run `python3 validate.py` to generate clearly labeled **reduced smoke-test** data,
or copy completed HPC outputs into `runs/`. The original data are read only.
Figures and tables go into a new timestamped `analysis/` folder.

For a single run, set `DATA_DIR` and `PREFIX` (or environment variables
`PIC_DATA_DIR`, `PIC_PREFIX`). For a paired experiment, set `COMPARISONS` below.
Missing optional files are reported; malformed required data raise a descriptive error.


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pic_analysis import read_config, load_table, clean_json, compare_runs

DATA_DIR = Path(os.environ.get('PIC_DATA_DIR', Path.cwd())).expanduser().resolve()
PREFIX = os.environ.get('PIC_PREFIX') or None  # Exact prefix, e.g. 'tsi_main_dt01_T60'.
FIT_WINDOW = None  # None: TSI [0,2], Weibel [0,1.6]; or (start,end).
# Each pair contains full output prefixes, including its run directory.
# Example: {'projection': ['runs/off/job/tsi_projection_off_dt01_T60',
#                          'runs/on/job/tsi_projection_on_dt01_T60']}
COMPARISONS = json.loads(os.environ.get('PIC_COMPARISONS', '{}'))

found = sorted(p.name[:-len('_diagnostics.csv')] for p in DATA_DIR.glob('*_diagnostics.csv')
               if not p.name.endswith(('_step_diagnostics.csv', '_adaptive_diagnostics.csv', '_failed_particle_diagnostics.csv')))
if PREFIX is None and len(found) == 1:
    PREFIX = found[0]
RUN_READY = PREFIX is not None
OUT = None

# ---------------------------------------------------------------------------
# Read one CSV from the selected output prefix
# Inputs:
#   suffix : str, table suffix; required : bool, reject missing/empty tables
# Output:
#   pandas.DataFrame : saved data, empty for an unavailable optional table
# Dependencies:
#   - pic_analysis.load_table; DATA_DIR and PREFIX configured above
# ---------------------------------------------------------------------------
def load(suffix, required=False):
    return load_table(DATA_DIR / PREFIX, suffix, required)

# ---------------------------------------------------------------------------
# Save and display a finished analysis figure
# Inputs:
#   fig : matplotlib Figure; name : str, output filename stem
# Output:
#   None; writes PNG and PDF files and closes the displayed figure
# Dependencies:
#   - matplotlib.pyplot as plt; OUT : active analysis directory
# ---------------------------------------------------------------------------
def save(fig, name):
    fig.savefig(OUT / f'{name}.png', dpi=180, bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ---------------------------------------------------------------------------
# Plot available diagnostic columns, omitting nonfinite and zero log values
# Inputs:
#   ax : matplotlib Axes; df : DataFrame; columns : (column,label) pairs
#   x : str, horizontal column; log : bool, absolute values on a log axis
# Output:
#   None; modifies ax, skips unavailable columns
# Dependencies:
#   - numpy as np, pandas as pd, matplotlib Axes
# ---------------------------------------------------------------------------
def curves(ax, df, columns, x='time', log=False):
    if x in df:
        for col, label in columns:
            if col in df:
                y = pd.to_numeric(df[col], errors='coerce')
                y = y.where(np.isfinite(y))
                if log:
                    y = y.abs().where(y.abs() > 0)
                if y.notna().any():
                    ax.plot(df[x], y, label=label)
    if log:
        ax.set_yscale('log')
    ax.set_xlabel(x)
    ax.grid(True, alpha=.3)
    if ax.lines:
        ax.legend(fontsize=8)

if RUN_READY:
    prefix_path = DATA_DIR / PREFIX
    diag_path = Path(f'{prefix_path}_diagnostics.csv')
    cfg = read_config(prefix_path)
    case = cfg.get('test_case', '')
    weibel = 'weibel' in case
    if not weibel and 'two_stream' not in case:
        raise ValueError(f'Unsupported case: {case}')
    mode = 'Bz_y_mode_abs' if weibel else 'E_mode_abs'
    required_columns = ['step', 'time', 'total_energy', 'nonlinear_converged', 'nonlinear_iterations', mode]
    diag = load_table(prefix_path, 'diagnostics', True, required_columns)
    step = load('step_diagnostics'); moments = load('velocity_moments')
    inner = load('particle_inner_history'); attempts = load('adaptive_diagnostics')
    OUT = DATA_DIR / 'analysis' / f"{PREFIX}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    OUT.mkdir(parents=True, exist_ok=False)
    print('Input:', diag_path, '\nCase:', case, '\nSaved figures and tables:', OUT)
    display(pd.Series({k: cfg.get(k, 'not recorded') for k in ['dt', 'n_steps', 'target_time', 'nx', 'ny', 'nz', 'spline_order', 'nyquist_projection', 'selective_particle_fallback', 'nonlinear_rtol', 'nonlinear_atol', 'particle_local_rtol', 'particle_local_atol']}).to_frame('configuration'))
else:
    print('No single run selected. Set DATA_DIR and PREFIX. Available prefixes:', found)
    print('Single-run plots skipped. This is not a validation pass or manuscript result.')


## Completion and convergence checks
A file being present does not prove that the run completed. These checks are separate from physical accuracy.

In [ ]:
if RUN_READY:
    summary={'prefix':PREFIX,'case':case,'configuration':cfg}
    expected_steps=int(cfg['n_steps']); target=float(cfg.get('target_time',expected_steps*float(cfg['dt'])))
    accepted=diag.loc[diag['step']>0].copy()
    summary.update(final_time=float(diag['time'].iloc[-1]),target_time=target,accepted_rows=len(accepted))
    summary['complete']=bool(len(diag)==expected_steps+1 and np.array_equal(diag['step'].to_numpy(),np.arange(expected_steps+1)) and np.isclose(diag['time'].iloc[-1],target,rtol=1e-10,atol=1e-10))
    summary['unconverged_accepted_steps']=int((accepted['nonlinear_converged']!=1).sum())
    if not attempts.empty:
        summary['nonaccepted_attempts']=int((attempts['outcome']!='accepted').sum())
        attempts.loc[attempts['outcome']!='accepted'].to_csv(OUT/'nonaccepted_attempts.csv',index=False)
    if not inner.empty:
        summary['failed_inner_evaluations']=int(((inner['particles_converged']!=1)|(inner['failed_particles']>0)).sum())
    verification=DATA_DIR/f'{PREFIX}_verification_summary.json'
    if verification.exists():
        previous=json.loads(verification.read_text())
        print('HPC screening:',previous.get('screening_pass'),'Issues:',previous.get('issues'))
        summary['hpc_screening_pass']=previous.get('screening_pass')
    display(pd.Series({k:v for k,v in summary.items() if k!='configuration'}).to_frame('value'))
    if not summary['complete']: print('INCOMPLETE RUN: figures show available data only.')


## Mode growth and energy
The fit is skipped with fewer than five finite positive samples. Even five points do not establish a reliable growth rate; inspect the chosen linear interval.

In [ ]:
if RUN_READY:
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    curves(axes[0],diag,[(mode,'measured mode'),('theory_mode_abs','finite-grid theory')],log=True)
    axes[0].set_ylabel('mode amplitude')
    window=FIT_WINDOW if FIT_WINDOW is not None else (0,1.6 if weibel else 2)
    time_eps=1e-12*max(1.0,abs(window[0]),abs(window[1]))
    mask=diag['time'].between(window[0]-time_eps,window[1]+time_eps)&np.isfinite(diag[mode])&(diag[mode]>0)
    fit=diag.loc[mask,['time',mode]]
    summary['growth_fit_samples']=len(fit); summary['growth_fit_window']=list(window)
    if len(fit)>=5 and fit['time'].nunique()>=5:
        slope,intercept=np.polyfit(fit['time'],np.log(fit[mode]),1)
        theory_values=pd.to_numeric(diag.get('theory_growth_rate', pd.Series(dtype=float)), errors='coerce')
        theory_values=theory_values[np.isfinite(theory_values)]
        theory=float(theory_values.iloc[0]) if len(theory_values) else float('nan')
        summary['growth_rate_fit']=float(slope); summary['growth_rate_theory']=theory
        summary['growth_rate_relative_error']=float(abs(slope-theory)/abs(theory)) if theory!=0 else None
        axes[0].plot(fit['time'],np.exp(intercept+slope*fit['time']),'--',label=f'fit gamma={slope:.6g}');axes[0].legend()
        print('Growth fit:',slope,'theory:',theory,'samples:',len(fit))
    else: print('Growth fit skipped: only',len(fit),'samples in',window)
    E0=float(diag['total_energy'].iloc[0])
    if not np.isfinite(E0) or E0==0: raise ValueError('Invalid initial energy')
    relative=(diag['total_energy']-E0)/abs(E0)
    summary['max_relative_energy_error']=float(np.abs(relative).max())
    axes[1].plot(diag['time'],relative); axes[1].set(xlabel='time',ylabel='signed relative total-energy change'); axes[1].grid(alpha=.3)
    save(fig,'01_mode_and_energy_error')
    fig,ax=plt.subplots(figsize=(8,4),layout='constrained')
    curves(ax,diag,[('kinetic_energy','kinetic'),('field_energy','field'),('total_energy','total')]);ax.set_ylabel('energy');save(fig,'02_energy_components')


## Gauss, gauge, chain rule and energy-exchange residuals

In [ ]:
if RUN_READY:
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    curves(axes[0],diag,[(k,k) for k in ['gauss_rms','gauss_max','gauge_rms','gauge_max']],log=True)
    axes[0].set_ylabel('absolute constraint residual')
    curves(axes[1],diag,[(k,k) for k in ['gauss_relative','gauge_relative']],log=True)
    axes[1].set_ylabel('relative constraint residual');save(fig,'03_constraints')
    if not step.empty:
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        curves(axes[0],step,[(k,k) for k in ['A_chain_abs_rms','A_chain_max_norm']],log=True)
        axes[0].set_ylabel('absolute chain-rule residual')
        curves(axes[1],step,[(k,k) for k in ['deposit_gather_work_residual','particle_energy_residual','field_energy_residual','delta_total_energy']],log=True)
        axes[1].set_ylabel('absolute work / energy residual');save(fig,'04_chain_and_work')
        fig,ax=plt.subplots(figsize=(9,4),layout='constrained')
        for k in ['particle_energy_residual','field_energy_residual','deposit_gather_work_residual','delta_total_energy']:
            if k in step:ax.plot(step['time'],step[k].cumsum()/abs(E0),label=k)
        ax.set(xlabel='time',ylabel='signed cumulative residual / |E(0)|');ax.legend(fontsize=8);ax.grid(alpha=.3);save(fig,'05_cumulative_energy_budget')
    maxima=[]
    for name,frame,columns in [('main',diag,['gauss_rms','gauss_max','gauge_rms','gauge_max']),('step',step,['A_chain_abs_rms','A_chain_max_norm','A_chain_work_defect','deposit_gather_work_residual','particle_energy_residual','field_energy_residual'])]:
        for k in columns:
            if k in frame:maxima.append({'source':name,'quantity':k,'max_absolute':frame[k].abs().max(),'nonfinite_rows':int((~np.isfinite(frame[k])).sum())})
    maxima=pd.DataFrame(maxima);maxima.to_csv(OUT/'residual_maxima.csv',index=False);display(maxima)


## All-particle velocity moments
These are mass-weighted statistics, not estimates from the limited particle sample. Transverse RMS means sqrt(〈v_y²〉+〈v_z²〉), not the standard deviation.

In [ ]:
if RUN_READY:
    if not moments.empty:
        moments=moments.copy();moments['vperp_rms']=np.hypot(moments['rms_vy'],moments['rms_vz'])
        moments.to_csv(OUT/'velocity_moments_with_vperp.csv',index=False)
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        curves(axes[0],moments,[(k,k) for k in ['rms_vy','rms_vz','vperp_rms']],log=True);axes[0].set_ylabel('velocity RMS')
        curves(axes[1],moments,[(k,k) for k in ['mean_vx','mean_vy','mean_vz']]);axes[1].set_ylabel('mean velocity')
        save(fig,'06_velocity_moments')
        summary['max_transverse_rms']=float(moments['vperp_rms'].max());summary['final_transverse_rms']=float(moments['vperp_rms'].iloc[-1])


## Outer iterations and full inner-map work
Per-step inner counts describe the final evaluation only. The inner-history file includes all recorded trial/line-search evaluations. The sum of their mean particle iteration counts measures work per particle, not an outer iteration count.

In [ ]:
if RUN_READY:
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    curves(axes[0],accepted,[('nonlinear_iterations','outer iterations'),('local_particle_iterations_mean','final evaluation: inner mean'),('local_particle_iterations_max','final evaluation: inner max')]);axes[0].set_ylabel('iterations')
    curves(axes[1],accepted,[(k,k) for k in ['anderson_steps','picard_fallback_steps','backtracked_steps','anderson_restarts']]);axes[1].set_ylabel('count per accepted step');save(fig,'07_solver_iterations')
    summary['outer_iterations_mean']=float(accepted['nonlinear_iterations'].mean()) if len(accepted) else None
    summary['outer_iterations_max']=float(accepted['nonlinear_iterations'].max()) if len(accepted) else None
    if not inner.empty:
        grouped=inner.groupby(['time_n','dt'],as_index=False).agg(map_evaluations=('map_evaluation','size'),sum_mean_particle_iterations=('local_iterations_mean','sum'),max_particle_iterations=('local_iterations_max','max'),max_scaled_particle_residual=('max_scaled_particle_residual','max'),failed_particle_records=('failed_particles','sum'))
        grouped['time']=grouped['time_n']+grouped['dt'];grouped.to_csv(OUT/'inner_work_by_interval.csv',index=False)
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        curves(axes[0],grouped,[('map_evaluations','all recorded map evaluations')]);axes[0].set_ylabel('evaluations per time interval')
        curves(axes[1],grouped,[('sum_mean_particle_iterations','sum of inner means'),('max_particle_iterations','largest inner count')]);axes[1].set_ylabel('particle iteration work');save(fig,'08_inner_work')
        summary['total_inner_map_evaluations']=len(inner)
        print('Grouped by (time_n,dt); repeated attempts at the same interval are combined.')


## Optional spectra and field representation diagnostics
Spectral power is reported as written by the solver; these sums are not labelled as physical field energy. Probe differences compare two field representations and are not automatically errors against an exact solution.

In [ ]:
if RUN_READY:
    spectrum=load('field_spectrum')
    if not spectrum.empty:
        columns=[k for k in spectrum if k.endswith('_power')]
        grouped=spectrum.groupby(['time','wavevector_class'])[columns].sum().reset_index()
        grouped.to_csv(OUT/'spectrum_by_wavevector_class.csv',index=False)
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        for label,group in grouped.groupby('wavevector_class'):
            for ax,prefix in zip(axes,['E','B']):
                power=group[[k for k in columns if k.startswith(prefix)]].sum(axis=1)
                ax.plot(group['time'],power.where(power>0),label=label)
        for ax,label in zip(axes,['electric','magnetic']):
            ax.set(xlabel='time',ylabel=f'summed {label} spectral power',yscale='log');ax.legend(fontsize=8);ax.grid(alpha=.3)
        save(fig,'09_field_spectra')
    probe=load('field_line_probe');knots=load('field_knot_jumps')
    for frame,columns,name in [(probe,['E_representation_difference','B_representation_difference'],'10_field_representation'),(knots,['A_jump','gradA_jump','E_grid_gather_jump','E_shape_phi_jump','B_grid_gather_jump','B_shape_A_jump'],'11_knot_jumps')]:
        cols=[k for k in columns if k in frame]
        if not frame.empty and cols:
            maxima=frame.groupby('time')[cols].agg(lambda x: x.abs().max()).reset_index()
            maxima.to_csv(OUT/f'{name}.csv',index=False)
            fig,ax=plt.subplots(figsize=(9,4),layout='constrained');curves(ax,maxima,[(k,k) for k in cols],log=True)
            ax.set_ylabel('maximum over saved probe points');save(fig,name)


## Particle sample, snapshot inventory and saved report
The sample is a limited subset, not an all-particle density plot. Full snapshots are listed with their actual saved times; filenames contain requested times, which can differ at large dt.

In [ ]:
if RUN_READY:
    particles=load('particle_sample')
    if not particles.empty:
        coord='y' if weibel else 'x'
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        for ax,v in zip(axes,['vx','vy']):
            ax.scatter(particles[coord],particles[v],s=1,alpha=.5,rasterized=True);ax.set(xlabel=coord,ylabel=v,title='Final saved particle sample')
        save(fig,'12_particle_sample')
    snapshots=[]
    for path in DATA_DIR.glob(f'{PREFIX}_particles_t*.csv'):
        try:
            first=pd.read_csv(path,nrows=1)
        except pd.errors.EmptyDataError:
            print('Empty snapshot:', path.name); continue
        if not first.empty:snapshots.append({'file':path.name,'actual_time':float(first['time'].iloc[0]),'requested_time':float(first['requested_time'].iloc[0]),'bytes':path.stat().st_size})
    if snapshots:
        inventory=pd.DataFrame(snapshots).sort_values('actual_time');inventory.to_csv(OUT/'snapshot_inventory.csv',index=False);display(inventory)
    failed=load('failed_particle_diagnostics')
    summary['saved_failed_particle_records']=len(failed)
    if not failed.empty:
        failed.to_csv(OUT/'failed_particle_records.csv',index=False);display(failed.head(20))
    # JSON null explicitly represents missing/nonfinite summary values.
    (OUT/'analysis_summary.json').write_text(json.dumps(clean_json(summary),indent=2,allow_nan=False)+'\n')
    pd.Series({k:v for k,v in summary.items() if not isinstance(v,(dict,list))}).to_csv(OUT/'analysis_summary.csv',header=['value'])
    print('All figures and tables saved in:',OUT)
    print('PNG figures:',len(list(OUT.glob('*.png'))),'PDF figures:',len(list(OUT.glob('*.pdf'))))


## Matched projection and spline comparisons
Only completed pairs with matching recorded controls are accepted. Mass-weighted moments equal charge-weighted moments for these electron runs because m=|q|. Finite-offset knot differences are regularity diagnostics, not errors against an exact solution.


In [ ]:
# Paired comparisons validate controls and completion before plotting.
for kind, prefixes in COMPARISONS.items():
    table, runs = compare_runs(prefixes, kind)
    OUT = DATA_DIR / 'analysis' / f"{kind}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    OUT.mkdir(parents=True, exist_ok=False)
    table.to_csv(OUT / f'{kind}_comparison.csv', index=False)
    (OUT / 'comparison.json').write_text(json.dumps(clean_json(table.to_dict('records')), indent=2, allow_nan=False) + '\n')
    display(table)
    fig, axes = plt.subplots(1, 2, figsize=(12,4), layout='constrained')
    for run in runs:
        cfg_pair = run['config']
        label = ('projection ' + cfg_pair['nyquist_projection']) if kind == 'projection' else ('r=' + cfg_pair['spline_order'])
        moments_pair = run['moments']
        vperp = np.hypot(moments_pair['rms_vy'], moments_pair['rms_vz'])
        axes[0].plot(moments_pair['time'], vperp.where(vperp > 0), label=label)
        if kind == 'spline':
            knots_pair = run['knots'].groupby('time')[['E_shape_phi_jump', 'gradA_jump']].max()
            for column, style in [('E_shape_phi_jump','-'), ('gradA_jump','--')]:
                y=knots_pair[column]
                axes[1].plot(knots_pair.index, y.where(y > 0), style, label=f'{label}: {column}')
        else:
            sample = load_table(run['prefix'], 'particle_sample', False, ('x', 'vy'))
            if not sample.empty:
                axes[1].scatter(sample['x'], sample['vy'], s=1, alpha=.5, rasterized=True, label=label)
    axes[0].set(xlabel='time', ylabel='all-particle transverse RMS', yscale='log')
    if kind == 'spline':
        axes[1].set(xlabel='time', ylabel='maximum finite-offset knot difference', yscale='log')
    else:
        axes[1].set(xlabel='x', ylabel='vy', title='Final saved particle sample')
    for ax in axes:
        ax.grid(alpha=.3)
        if ax.lines or ax.collections: ax.legend(fontsize=8)
    save(fig, kind + '_comparison')
if not COMPARISONS:
    print('No paired comparisons selected. Configure COMPARISONS to analyze the new experiments.')
